# Galaxy morphology classification with AstroLens Linformer

A tutorial-scale training example for `astrolens.models.linformer.Linformer`
(Lin et al., 2021, https://arxiv.org/abs/2110.01024), using
[`UniverseTBD/mmu_gz10`](https://huggingface.co/datasets/UniverseTBD/mmu_gz10) —
a MultimodalUniverse-formatted copy of **Galaxy10 DECals** (17,736 galaxies,
10 discrete morphology classes, no vote-fraction preprocessing required).

The paper itself trains on the full Galaxy Zoo 2 dataset with an 8-class scheme
(round/in-between/cigar-shaped elliptical, edge-on, barred/unbarred spiral,
irregular, merger) derived from Hart et al. 2016 vote-fraction thresholds —
155,951 images, 64/16/20 split, 200 epochs. That exact label derivation isn't
reconstructable from `mwalmsley/gz2` on Hugging Face (it lacks the vote
fractions needed for the "odd feature" branch that separates irregular/merger),
and reproducing it at full scale is out of scope for a tutorial notebook. **To
reproduce the paper exactly**, use the original authors' repository and its
precomputed labels:
[`sliao-mi-luku/Galaxy-Zoo-Classification`](https://github.com/sliao-mi-luku/Galaxy-Zoo-Classification)
(`gz2_data/gz2_{train,valid,test}.csv` — galaxy ID → `label1` in 0-7, matching
the paper's split sizes almost exactly).

This notebook instead demonstrates the same architecture and training loop on
a smaller, self-contained dataset with ready-made discrete labels, split
70% train / 10% val / 20% test.

## Install example-only dependencies

Not part of AstroLens' core install (`requirements.txt`) — only needed for this example.

In [1]:
!pip install -q datasets torchvision scikit-learn

## Imports

In [2]:
import io

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from datasets import load_dataset
from PIL import Image
from sklearn.model_selection import train_test_split

import astrolens

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Load the dataset and split 70/10/20

Galaxy10 DECals' standard 10 classes (astroNN convention): disturbed, merging,
round smooth, in-between round smooth, cigar-shaped smooth, barred spiral,
unbarred tight spiral, unbarred loose spiral, edge-on without bulge, edge-on
with bulge.

In [3]:
IMG_SIZE = 224
BATCH_SIZE = 64
CLASS_NAMES = [
    "disturbed",
    "merging",
    "round_smooth",
    "in_between_round_smooth",
    "cigar_shaped_smooth",
    "barred_spiral",
    "unbarred_tight_spiral",
    "unbarred_loose_spiral",
    "edge_on_no_bulge",
    "edge_on_with_bulge",
]
NUM_CLASSES = len(CLASS_NAMES)

gz10 = load_dataset("UniverseTBD/mmu_gz10", split="train")
labels = gz10["gz10_label"]

train_idx, rest_idx = train_test_split(
    range(len(gz10)), train_size=0.7, stratify=labels, random_state=0
)
val_idx, test_idx = train_test_split(
    rest_idx,
    train_size=1 / 3,  # 1/3 of the remaining 30% -> 10% val, 20% test
    stratify=[labels[i] for i in rest_idx],
    random_state=0,
)

# per-channel mean/std computed from a 2000-image sample of the gz10 train split
IMAGE_MEAN = [0.1675, 0.1625, 0.1586]
IMAGE_STD = [0.1288, 0.1178, 0.1109]

train_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.RandomRotation(90),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(IMAGE_MEAN, IMAGE_STD),
    ]
)
eval_transform = transforms.Compose(
    [
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGE_MEAN, IMAGE_STD),
    ]
)


class GZ10Dataset(Dataset):
    """Map-style wrapper around an index subset of the HF split, applying transform lazily."""

    def __init__(self, hf_split, indices, transform):
        self.hf_split = hf_split
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        example = self.hf_split[self.indices[i]]
        image = Image.open(io.BytesIO(example["rgb_image"]["bytes"])).convert("RGB")
        return self.transform(image), example["gz10_label"]


train_dataset = GZ10Dataset(gz10, train_idx, train_transform)
val_dataset = GZ10Dataset(gz10, val_idx, eval_transform)
test_dataset = GZ10Dataset(gz10, test_idx, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4)

len(train_dataset), len(val_dataset), len(test_dataset)

Resolving data files:   0%|          | 0/921 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/766 [00:00<?, ?it/s]

(12415, 1773, 3548)

## Create the model

Uses `Linformer`'s default configuration, matching the paper (Lin et al., 2021):
`patch_size=28, dim=128, depth=12, heads=8, k=64`.

In [4]:
model = astrolens.create_model(
    "linformer",
    img_size=IMG_SIZE,
    in_chans=3,
    num_classes=NUM_CLASSES,
).to(device)

sum(p.numel() for p in model.parameters())

2790634

## Train

In [ ]:
MAX_EPOCHS = 100
LR = 3e-4
# StepLR schedule from the paper: decay LR by gamma every step_size epochs
STEP_SIZE = 5
GAMMA = 0.9

# inverse-frequency class weights from the train split, following the paper's
# use of class-weighted cross-entropy to counter GZ10's class imbalance
train_counts = torch.bincount(
    torch.tensor([labels[i] for i in train_idx]), minlength=NUM_CLASSES
).float()
class_weights = (train_counts.sum() / (NUM_CLASSES * train_counts)).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)


def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, correct, count = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            count += images.size(0)

    return total_loss / count, correct / count


for epoch in range(1, MAX_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(
        f"epoch {epoch}/{MAX_EPOCHS} "
        f"train_loss={train_loss:.3f} train_acc={train_acc:.3f} "
        f"val_loss={val_loss:.3f} val_acc={val_acc:.3f} "
        f"lr={scheduler.get_last_lr()[0]:.2e}"
    )

## Evaluate on the held-out test split

In [6]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        y_pred.extend(logits.argmax(dim=1).cpu().tolist())
        y_true.extend(labels.tolist())

test_acc = accuracy_score(y_true, y_pred)
test_f1_macro = f1_score(y_true, y_pred, average="macro")
print(f"test_acc={test_acc:.3f} test_f1_macro={test_f1_macro:.3f}\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

test_acc=0.792 test_f1_macro=0.769

                         precision    recall  f1-score   support

              disturbed       0.49      0.38      0.43       216
                merging       0.81      0.82      0.81       371
           round_smooth       0.93      0.93      0.93       529
in_between_round_smooth       0.86      0.93      0.90       405
    cigar_shaped_smooth       0.68      0.73      0.71        67
          barred_spiral       0.83      0.76      0.79       409
  unbarred_tight_spiral       0.65      0.75      0.69       366
  unbarred_loose_spiral       0.65      0.64      0.64       525
       edge_on_no_bulge       0.91      0.86      0.88       285
     edge_on_with_bulge       0.89      0.91      0.90       375

               accuracy                           0.79      3548
              macro avg       0.77      0.77      0.77      3548
           weighted avg       0.79      0.79      0.79      3548



## Next steps

- Raise `MAX_EPOCHS` or add a learning-rate schedule / early stopping for a longer run.
- To reproduce the paper's actual 8-class Galaxy Zoo 2 result (155,951 images,
  200 epochs, class-weighted loss), follow
  [`sliao-mi-luku/Galaxy-Zoo-Classification`](https://github.com/sliao-mi-luku/Galaxy-Zoo-Classification)
  directly — it ships the precomputed `label1` splits this notebook doesn't
  attempt to rederive.